# 🧪 Pipeline Cào Dữ Liệu Thuốc Biệt Dược (Bronze Zone — Data Lakehouse)

Notebook này đóng gói toàn bộ quy trình cào và đóng gói dữ liệu thô từ website **Thuốc Biệt Dược** (`thuocbietduoc.com.vn`), đưa vào **Tầng Bronze (Raw Zone)** với độ tin cậy cao nhất (**`trust_score = 1.0`**).

### 📋 Quy Chuẩn Tầng Bronze:
- **Nguồn dữ liệu (`source_name`)**: `Thuốc Biệt Dược`
- **Mức độ tin cậy (`trust_score`)**: `1.0` (Dữ liệu y khoa chính thống)
- **Định dạng file**: Raw JSON chứa thông tin trích xuất + HTML gốc.
- **Metadata bắt buộc**: `source_url`, `ingestion_timestamp`, `file_hash` (SHA-256), `lineage_hash`.

## 1. Cấu Hình & Thiết Lập Môi Trường (Cần Chạy Đầu Tiên)

In [ ]:
import os
import sys
import time
import math
import json
import re
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'thuocbietduoc_scraper':
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / 'bronze' / 'thuocbietduoc'
QUEUE_FILE = PROJECT_ROOT / 'thuocbietduoc_scraper' / 'url_queue_thuocbietduoc.json'

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_FILE.parent.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://thuocbietduoc.com.vn'
AZ_SEARCH_URL = 'https://thuocbietduoc.com.vn/thuoc/drgsearch.aspx'
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/121.0'
]

REQUEST_DELAY_MIN = 0.5
REQUEST_DELAY_MAX = 1.2
MAX_RETRIES = 3
TIMEOUT_SECONDS = 15
SOURCE_NAME = 'Thuốc Biệt Dược'
TRUST_SCORE = 1.0

print('✅ Đã khởi tạo xong Cấu Hình Thuốc Biệt Dược!')
print(f'📂 Thư mục chứa dữ liệu Bronze: {BRONZE_DIR}')
print(f'📋 File Quản lý Queue:           {QUEUE_FILE}')

## 2. Giai Đoạn 1: Khám Phá URL (Hỗ Trợ Cả Quét A-Z & Danh Mục Y Khoa)

Hỗ trợ 2 chế độ quét:
1. **`mode='category'`**: Duyệt qua 35 Danh mục Y Khoa (`/nhom-thuoc-{id}-0/thuoc.aspx`).
2. **`mode='az'`**: Duyệt qua bảng chữ cái **A đến Z** (`/thuoc/drgsearch.aspx?type=1&drgname={A..Z}`).

In [ ]:
if 'QUEUE_FILE' not in globals():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'thuocbietduoc_scraper' else Path.cwd()
    QUEUE_FILE = PROJECT_ROOT / 'thuocbietduoc_scraper' / 'url_queue_thuocbietduoc.json'
    BRONZE_DIR = PROJECT_ROOT / 'bronze' / 'thuocbietduoc'
    BASE_URL = 'https://thuocbietduoc.com.vn'
    AZ_SEARCH_URL = 'https://thuocbietduoc.com.vn/thuoc/drgsearch.aspx'
    USER_AGENTS = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36']
    REQUEST_DELAY_MIN, REQUEST_DELAY_MAX, MAX_RETRIES, TIMEOUT_SECONDS = 0.5, 1.2, 3, 15
    SOURCE_NAME, TRUST_SCORE = 'Thuốc Biệt Dược', 1.0

def load_queue():
    if QUEUE_FILE.exists():
        try:
            with open(QUEUE_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f'⚠️ Warning: Lỗi đọc file queue ({e}), khởi tạo mới.')
    return {}

def save_queue(queue):
    with open(QUEUE_FILE, 'w', encoding='utf-8') as f:
        json.dump(queue, f, ensure_ascii=False, indent=2)

def get_headers():
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    }

def fetch_category_page(cat_id: int):
    url = f"{BASE_URL}/nhom-thuoc-{cat_id}-0/thuoc.aspx"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                html = resp.text
                drug_links = set(re.findall(r'href=["\'](https://thuocbietduoc\.com\.vn/thuoc-\d+/[^"\']+\.aspx)["\']', html))
                rel_links = set(re.findall(r'href=["\'](/thuoc-\d+/[^"\']+\.aspx)["\']', html))
                for r in rel_links:
                    drug_links.add(f"{BASE_URL}{r}")
                return list(drug_links)
        except Exception:
            pass
        time.sleep(attempt * 1.2)
    return []

def fetch_az_page(letter: str, page: int = 1):
    url = f"{AZ_SEARCH_URL}?type=1&drgname={letter}&page={page}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                html = resp.text
                drug_links = set(re.findall(r'href=["\'](https://thuocbietduoc\.com\.vn/thuoc-\d+/[^"\']+\.aspx)["\']', html))
                rel_links = set(re.findall(r'href=["\'](/thuoc-\d+/[^"\']+\.aspx)["\']', html))
                for r in rel_links:
                    drug_links.add(f"{BASE_URL}{r}")
                page_nums = [int(p) for p in re.findall(r'page=(\d+)', html) if p.isdigit()]
                max_p = max(page_nums) if page_nums else 1
                return list(drug_links), max_p
        except Exception:
            pass
        time.sleep(attempt * 1.2)
    return [], 1

def discover_urls(mode='category', cat_range=range(1, 36), letters=None):
    queue = load_queue()
    initial_count = len(queue)
    new_urls_count = 0

    if mode == 'az':
        if not letters:
            letters = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
        print(f'🔍 [Chế độ A-Z] Bắt đầu quét URL cho các chữ cái: {", ".join(letters)}')
        for letter in letters:
            print(f'\n---> Quét chữ cái: "{letter}"')
            first_links, max_pages = fetch_az_page(letter, page=1)
            for u in first_links:
                if u not in queue:
                    queue[u] = {'status': 'pending', 'discovered_at': datetime.now(timezone.utc).isoformat(), 'attempts': 0, 'error': None}
                    new_urls_count += 1
            print(f'  📊 Chữ "{letter}": Tìm thấy {len(first_links)} thuốc trên trang 1 (Tổng {max_pages} trang)')
            save_queue(queue)
    else:
        print(f'🔍 [Chế độ Danh Mục] Bắt đầu quét URL cho {len(cat_range)} danh mục Thuốc Biệt Dược...')
        pbar = tqdm(cat_range, desc='Scanning Categories', unit='cat')
        for cat_id in pbar:
            time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))
            drug_links = fetch_category_page(cat_id)
            for full_url in drug_links:
                if full_url not in queue:
                    queue[full_url] = {
                        'status': 'pending',
                        'discovered_at': datetime.now(timezone.utc).isoformat(),
                        'category_id': cat_id,
                        'attempts': 0,
                        'error': None
                    }
                    new_urls_count += 1
            if cat_id % 5 == 0:
                save_queue(queue)

    save_queue(queue)
    print('\n==========================================')
    print('✅ Hoàn thành khám phá URL Thuốc Biệt Dược!')
    print(f'➕ URL mới thêm vào: {new_urls_count}')
    print(f'📋 Tổng số URL trong Queue: {len(queue)}')
    print(f'💾 Queue lưu tại: {QUEUE_FILE}')

### 🚀 Thực Thi Khám Phá URL Thuốc Biệt Dược

- Quét toàn bộ **35 Danh mục Y Khoa**: `discover_urls(mode='category', cat_range=range(1, 36))`.
- Hoặc quét toàn bộ **Bảng Chữ Cái A đến Z**: `discover_urls(mode='az', letters=None)`.

In [ ]:
# Thực thi quét toàn bộ 35 danh mục Y Khoa (hoặc đổi mode='az' để quét theo A-Z)
discover_urls(mode='category', cat_range=range(1, 36))

## 3. Giai Đoạn 2: Cào Chi Tiết & Đóng Gói Dữ Liệu Tầng Bronze (Bronze Ingestion)

Hàm `scrape_bronze()` sẽ đọc danh sách URL `pending` từ queue, tải bài viết chi tiết, trích xuất cấu trúc thông tin y khoa, đính kèm 5 trường Metadata tầng Bronze và lưu file raw JSON vào `bronze/thuocbietduoc/`.

In [ ]:
def clean_slug_filename(url: str) -> str:
    slug = url.split('/')[-1].replace('.aspx', '')
    slug_clean = re.sub(r'[^a-zA-Z0-9_-]', '_', slug)
    url_hash = hashlib.sha256(url.encode('utf-8')).hexdigest()[:8]
    return f"{slug_clean}_{url_hash}.json"

def parse_tbd_page(html: str):
    soup = BeautifulSoup(html, 'html.parser')
    h1 = soup.find('h1')
    title = h1.text.strip() if h1 else ''
    
    parsed_data = {
        'title': title,
        'fields': {},
        'full_text': soup.get_text(separator='\n', strip=True)
    }
    
    rows = soup.find_all('tr')
    for r in rows:
        tds = r.find_all(['td', 'th'])
        if len(tds) == 2:
            key = tds[0].text.strip().rstrip(':')
            val = tds[1].text.strip()
            if key and val:
                parsed_data['fields'][key] = val
    
    headings = soup.find_all(['h2', 'h3', 'b', 'strong'])
    for h in headings:
        htext = h.text.strip().rstrip(':')
        if any(keyword in htext for keyword in ['Chỉ định', 'Chống chỉ định', 'Liều dùng', 'Cách dùng', 'Thành phần', 'Tương tác', 'Tác dụng phụ', 'Thận trọng', 'Bảo quản']):
            next_node = h.next_sibling
            content_buf = []
            count = 0
            while next_node and count < 5:
                if hasattr(next_node, 'get_text'):
                    t = next_node.get_text(strip=True)
                    if t:
                        content_buf.append(t)
                next_node = next_node.next_sibling
                count += 1
            if content_buf:
                parsed_data['fields'][htext] = ' '.join(content_buf)
                
    return parsed_data

def fetch_product_data(url: str):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                html = resp.text
                parsed = parse_tbd_page(html)
                parsed['html_raw'] = html
                return parsed, html
            elif resp.status_code == 404:
                return None, 'HTTP 404 Not Found'
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, str(e)
        time.sleep(attempt * 1.5)
    return None, 'Quá số lần thử lại'

def scrape_bronze(limit: int = None, retry_failed: bool = False):
    queue = load_queue()
    target_urls = [url for url, info in queue.items() if info.get('status') == 'pending' or (retry_failed and info.get('status') == 'failed')]

    if not target_urls:
        print('✨ Không có URL nào ở trạng thái pending trong Queue!')
        return

    if limit:
        target_urls = target_urls[:limit]

    print(f'🚀 Bắt đầu cào dữ liệu Bronze cho {len(target_urls)} sản phẩm Thuốc Biệt Dược (Limit: {limit or "Tất cả"})')
    print(f'📂 Lưu file tại: {BRONZE_DIR}')

    success_count = 0
    fail_count = 0
    pbar = tqdm(target_urls, desc='Scraping Bronze Zone', unit='doc')

    for i, url in enumerate(pbar):
        queue[url]['attempts'] = queue[url].get('attempts', 0) + 1
        parsed_data, raw_or_err = fetch_product_data(url)

        if parsed_data:
            raw_data_str = json.dumps(parsed_data, ensure_ascii=False)
            file_hash = hashlib.sha256(raw_data_str.encode('utf-8')).hexdigest()
            ingestion_time = datetime.now(timezone.utc).isoformat()
            lineage_hash = hashlib.sha256(f'{url}_{file_hash}_{ingestion_time}'.encode('utf-8')).hexdigest()

            bronze_record = {
                'source_name': SOURCE_NAME,
                'trust_score': TRUST_SCORE,
                'source_url': url,
                'ingestion_timestamp': ingestion_time,
                'file_hash': file_hash,
                'lineage_hash': lineage_hash,
                'raw_data': parsed_data
            }

            filename = clean_slug_filename(url)
            file_path = BRONZE_DIR / filename
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(bronze_record, f, ensure_ascii=False, indent=2)

            queue[url]['status'] = 'done'
            queue[url]['scraped_at'] = ingestion_time
            queue[url]['file_path'] = str(file_path)
            queue[url]['error'] = None
            success_count += 1
        else:
            queue[url]['status'] = 'failed'
            queue[url]['error'] = str(raw_or_err)
            fail_count += 1

        if (i + 1) % 5 == 0:
            save_queue(queue)
        time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))

    save_queue(queue)
    print('\n==========================================')
    print('🎉 Hoàn thành phiên cào Bronze Thuốc Biệt Dược!')
    print(f'✅ Thành công: {success_count}')
    print(f'❌ Thất bại: {fail_count}')
    print(f'💾 Tổng số file tại Bronze: {len(list(BRONZE_DIR.glob("*.json")))}')


### 🚀 Thực Thi Cào Dữ Liệu Bronze Thuốc Biệt Dược (Toàn Bộ Sản Phẩm)

- `scrape_bronze(limit=None)`: Cào tất cả các sản phẩm đang có trong hàng đợi Queue.

In [ ]:
# Thực thi cào dữ liệu Bronze cho TOÀN BỘ sản phẩm trong queue (đặt limit=None)
scrape_bronze(limit=None)

## 4. Kiểm Tra & Trích Xuất Mẫu Dữ Liệu Bronze (Data Verification)

Đọc kiểm tra 1 file JSON ngẫu nhiên trong `bronze/thuocbietduoc/` để xác minh 5 trường metadata bắt buộc (`trust_score = 1.0`).

In [ ]:
bronze_files = list(BRONZE_DIR.glob('*.json'))
if bronze_files:
    sample_file = bronze_files[0]
    print(f'📄 Đọc kiểm tra file mẫu: {sample_file.name}\n')
    with open(sample_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print('--- 1. METADATA TẦNG BRONZE ---')
    print(f'Source Name : {data.get("source_name")}')
    print(f'Trust Score : {data.get("trust_score")}')
    print(f'Source URL  : {data.get("source_url")}')
    print(f'Timestamp   : {data.get("ingestion_timestamp")}')
    print(f'File Hash   : {data.get("file_hash")}')
    
    raw = data.get('raw_data', {})
    print('\n--- 2. THÔNG TIN SẢN PHẨM THÔ ---')
    print(f'Tiêu đề     : {raw.get("title")}')
    fields = raw.get('fields', {})
    for k, v in list(fields.items())[:10]:
        print(f'{k:<12}: {str(v)[:100]}')
else:
    print('⚠️ Chưa có file nào trong thư mục bronze/thuocbietduoc/')